# Train the learned-optimiser transformer on Kaggle

Clones the repo, runs `src.train`, then `src.validate`. Unlike notebooks 01-04 this one is
**not** self-contained — it is the Docker pipeline (`src/`) executed on a Kaggle GPU, so the
code that runs is exactly the code in the repo, with no copy to drift out of sync.

## What this run is testing

The angle **scale normalisation** (`src/angles.py`). The phase separator applies
`exp(i*gamma*E)`, and for this `J` the spectrum spans about 39, so the phase completes a full
revolution by `gamma ~ 2*pi/39 ~ 0.16`. The mixer is periodic in `beta` with period `pi`. The
two natural scales are ~20x apart, and the pipeline previously treated the 10-vector as
homogeneous — which mis-scaled the random init, the Adam step size, and (worst) saturated the
gamma half of the gradient features against a shared clip.

Everything upstream of the simulator now works in normalised units. **§3 prints the measured
scale before training starts** — check it before spending GPU hours on the rest.

## Before you run

1. **Settings -> Accelerator -> GPU** (P100 or T4).
2. **Settings -> Internet -> On.** Required: this notebook clones from GitHub.
3. The repo is **private**, so give Kaggle a credential of its own — your local SSH key is
   not present in a Kaggle container. Under *Add-ons -> Secrets*, add **either**:

   | secret | value | notes |
   |---|---|---|
   | `GITHUB_TOKEN` | fine-grained PAT, **Contents: Read** on `brkdrd/sberchall` | recommended — read-only, single repo, revocable on its own |
   | `SSH_KEY` | an SSH **private** key authorised on the repo | use a deploy key, not your personal key; `base64 -w0 ~/.ssh/id_ed25519` if Kaggle mangles the newlines |

   `SSH_KEY` wins if both are attached. With neither, the clone falls back to anonymous, which
   only works if you make the repo public.

`J.npy` and `h_train.npy` are tracked in the repo, so no Kaggle dataset needs attaching.

## 1. Environment

In [ ]:
import os, sys, subprocess, time, json, shutil, textwrap
from pathlib import Path

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
      or "NO GPU -- turn on the accelerator in Settings")
import torch
print(f"torch {torch.__version__} | cuda available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("\nWARNING: training on CPU will not finish in a Kaggle session.")

## 2. Configuration

`TRAIN_HOURS` is the real control. `src/train.py` writes `best.pt` at every eval
(`--eval-every`), so the run is safe to stop at any point — the notebook enforces the budget by
terminating the process and then validating whatever checkpoint exists. That means you can set
`ITERS` optimistically and let the clock decide, rather than guessing the throughput up front.

In [ ]:
REPO_HTTPS = "github.com/brkdrd/sberchall.git"      # used with a GITHUB_TOKEN secret
REPO_SSH   = "git@github.com:brkdrd/sberchall.git"  # used with an SSH_KEY secret
BRANCH = "main"
WORK   = Path("/kaggle/working")
SRC    = WORK / "sberchall"

TRAIN_HOURS = 6.0     # wall-clock budget for training; best.pt survives an early stop
ITERS       = 12000   # upper bound; the clock usually binds first
BATCH       = 128
STEPS       = 8       # rollout length
LR          = 3e-4
EVAL_EVERY  = 500     # also the checkpoint interval -- lower it if you expect to be cut off
EVAL_RESTARTS = 16

VAL_RESTARTS = 256    # inference stack for the final number
VAL_POLISH   = 100
VAL_TOP_M    = 16

QUICK = False         # True -> ~5 min end-to-end smoke test of the whole pipeline
if QUICK:
    TRAIN_HOURS, ITERS, EVAL_EVERY, EVAL_RESTARTS = 0.08, 200, 100, 4
    VAL_RESTARTS, VAL_POLISH = 16, 20

print(json.dumps({k: v for k, v in globals().items()
                  if k.isupper() and isinstance(v, (int, float, str, bool))}, indent=2))

## 3. Clone the repo

In [ ]:
def _secret(name):
    """Read a Kaggle secret, or None if it is not attached."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None


def _ssh_env(key):
    """Write an SSH private key from the secret and return env for git.

    Kaggle secrets can mangle multi-line values, so a base64-encoded key is accepted too:
        base64 -w0 ~/.ssh/id_ed25519
    """
    import base64, binascii
    if "PRIVATE KEY" not in key:                      # looks encoded -> try base64
        try:
            key = base64.b64decode(key).decode()
        except (binascii.Error, UnicodeDecodeError):
            pass
    if "PRIVATE KEY" not in key:
        raise RuntimeError("SSH_KEY is neither a PEM private key nor valid base64 of one")
    d = Path.home() / ".ssh"
    d.mkdir(mode=0o700, exist_ok=True)
    kp = d / "id_kaggle"
    kp.write_text(key.rstrip("\n") + "\n")           # OpenSSH requires the trailing newline
    kp.chmod(0o600)
    return dict(os.environ, GIT_SSH_COMMAND=(
        f"ssh -i {kp} -o IdentitiesOnly=yes -o StrictHostKeyChecking=accept-new"))


def clone_repo():
    """SSH key -> PAT over HTTPS -> anonymous. Returns (url_for_logging, env)."""
    k = _secret("SSH_KEY")
    if k:
        print("auth: SSH_KEY secret")
        return REPO_SSH, _ssh_env(k)
    t = _secret("GITHUB_TOKEN")
    if t:
        print("auth: GITHUB_TOKEN secret (https)")
        return f"https://{t}@{REPO_HTTPS}", os.environ.copy()
    print("auth: none found -- trying an anonymous clone (works only if the repo is public)")
    return f"https://{REPO_HTTPS}", os.environ.copy()


if SRC.exists():
    shutil.rmtree(SRC)                      # always start from a clean checkout
url, env = clone_repo()
r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, str(SRC)],
                   capture_output=True, text=True, env=env)
if r.returncode:
    err = r.stderr
    for sec in ("SSH_KEY", "GITHUB_TOKEN"):
        v = _secret(sec)
        if v:
            err = err.replace(v, "***")
    raise RuntimeError(
        err + "\n\nAttach a credential under Add-ons -> Secrets:\n"
        "  SSH_KEY      a deploy/user SSH private key (base64 is fine), or\n"
        "  GITHUB_TOKEN a fine-grained PAT with Contents: Read on this repo")

os.chdir(SRC)
sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
msg = subprocess.run(["git", "log", "-1", "--pretty=%s"], capture_output=True, text=True).stdout.strip()
print(f"cloned {BRANCH} @ {sha}: {msg}")
print("data:", sorted(p.name for p in Path("data/raw").glob("*.npy")))
assert Path("data/raw/J.npy").exists() and Path("data/raw/h_train.npy").exists()


## 4. Check the measured angle scale

The whole point of this run. `angle_scale` is derived from the actual spectrum, not
hardcoded — if these numbers look wrong, stop here rather than burning GPU hours.

Expect a span near **39**, a gamma unit near **0.16 rad**, a beta unit of **pi/2 ~ 1.571 rad**,
and a ratio near **10x**.

In [ ]:
sys.path.insert(0, str(SRC))
import numpy as np, torch
from src.qaoa_ref import QAOA
from src.angles import angle_scale, energy_span

dev = "cuda" if torch.cuda.is_available() else "cpu"
_sim = QAOA(np.load("data/raw/J.npy"), device=dev)
_h = torch.tensor(np.load("data/raw/h_train.npy"), dtype=torch.float32, device=dev)
_span = energy_span(_sim, _h)
_s = angle_scale(_sim, _h, device=dev)
print(f"energy span      : {_span:.3f}")
print(f"gamma unit       : {_s[0].item():.4f} rad   (one full phase revolution)")
print(f"beta  unit       : {_s[-1].item():.4f} rad   (half the mixer period)")
print(f"ratio            : {_s[-1].item()/_s[0].item():.1f}x")
print(f"\nold init sampled gamma uniformly on (-{np.pi/2:.3f}, {np.pi/2:.3f}) rad")
print(f"  -> {(np.pi/2)/_s[0].item():.1f}x wider than one revolution, per gamma dimension")
print(f"  -> useful fraction of the 5-D gamma box: {(_s[0].item()/(np.pi/2))**5:.2e}")
del _sim, _h, _s
if dev == "cuda":
    torch.cuda.empty_cache()

## 5. Train

Streams `src/train.py`'s stdout live. Two lines matter:

- the **startup line** repeats the scale above, confirming training uses it;
- **`P start`** on the log lines is the quality of the *random initial* angles, before the
  model contributes anything. Under the old parameterisation the init sampled the wrapped
  regime; if normalisation does what the arithmetic says, `P start` should be visibly higher
  from iteration 1 — that part is independent of any learning and is the cleanest early read.

`eval` lines are the real metric: mean best-of-K P(ground) on `h_train`, comparable with the
leaderboard and with notebooks 03/04.

In [ ]:
def stream(cmd, budget_s=None, tag=""):
    """Run a subprocess, echo stdout live, stop it at the budget. Returns (rc, seconds)."""
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, cwd=str(SRC))
    try:
        for line in p.stdout:
            print(line, end="", flush=True)
            if budget_s and time.time() - t0 > budget_s:
                print(f"\n[{tag}] wall-clock budget reached -- stopping; best.pt is already saved")
                p.terminate()
                try:
                    p.wait(timeout=60)
                except subprocess.TimeoutExpired:
                    p.kill()
                break
    except KeyboardInterrupt:
        p.terminate(); raise
    p.wait()
    return p.returncode, time.time() - t0


OUT = SRC / "runs" / "norm"
rc, secs = stream([sys.executable, "-m", "src.train",
                   "--out-dir", str(OUT),
                   "--iters", str(ITERS), "--batch", str(BATCH), "--steps", str(STEPS),
                   "--lr", str(LR), "--eval-every", str(EVAL_EVERY),
                   "--eval-restarts", str(EVAL_RESTARTS)],
                  budget_s=TRAIN_HOURS * 3600, tag="train")
print(f"\ntraining stopped after {secs/60:.1f} min (rc={rc})")
ck = OUT / "best.pt"
assert ck.exists(), "no checkpoint written -- training died before the first eval"
_c = torch.load(ck, map_location="cpu", weights_only=False)
print(f"best.pt: iter {_c['iter']}, eval metric {_c['metric']:.5f}, "
      f"angle_scale {'present' if _c.get('angle_scale') is not None else 'MISSING'}")
del _c

## 6. Validate

The full inference stack — best-of-K rollouts, then Adam polish on the top-M candidates with
selection *after* polishing. Reports the number to compare against everything else, plus
whether it fits the 10-minute inference budget.

Reference points on `h_train`:

| | mean P(ground) |
|---|---|
| transformer, pre-normalisation | 0.28165 |
| CMA-ES (nb 04) | 0.27725 |
| massive multistart (nb 03) | 0.32396 |
| leaderboard #10 / #1 | 0.34882 / 0.81468 |

In [ ]:
rc, secs = stream([sys.executable, "-m", "src.validate",
                   "--ckpt", str(ck),
                   "--restarts", str(VAL_RESTARTS), "--polish", str(VAL_POLISH),
                   "--top-m", str(VAL_TOP_M),
                   "--out", str(SRC / "runs" / "submission_train.csv")], tag="validate")
print(f"\nvalidation finished in {secs/60:.1f} min (rc={rc})")

## 7. Collect outputs

In [ ]:
import csv
sub = SRC / "runs" / "submission_train.csv"
for f in (sub, ck, OUT / "config.json"):
    if f.exists():
        dst = WORK / f.name
        if f.resolve() != dst.resolve():
            shutil.copy(f, dst)
        print(f"{dst}  ({dst.stat().st_size/1024:.0f} KiB)")

rows = list(csv.reader(open(sub)))[1:]
A = np.array([[float(x) for x in r[1:]] for r in rows], dtype=np.float32)
sim = QAOA(np.load(SRC / "data/raw/J.npy"), device=dev)
h = torch.tensor(np.load(SRC / "data/raw/h_train.npy"), dtype=torch.float32, device=dev)
with torch.no_grad():
    p = sim.p_ground(h, torch.tensor(A[:, :5], device=dev),
                     torch.tensor(A[:, 5:], device=dev)).cpu().numpy()
print(f"\nre-scored submission: mean {p.mean():.5f}  median {np.median(p):.5f}  "
      f"min {p.min():.5f}  max {p.max():.5f}")
print(f"distinct rows: {len(np.unique(np.round(A, 6), axis=0))}/{len(A)}  "
      f"(constant submissions score 0)")
print(f"\nvs pre-normalisation transformer 0.28165 : {p.mean()/0.28165:.3f}x")
print(f"vs massive multistart          0.32396 : {p.mean()/0.32396:.3f}x")

g = A[:, :5]
print(f"\ngamma actually used: |g| mean {np.abs(g).mean():.4f}, max {np.abs(g).max():.4f} rad")
print(f"  (init box was +-{2*np.pi/_span:.4f} rad; larger means the model walked outward, "
      f"which it is free to do)")

## 8. Reading the result

- **`P start` barely moved from the old runs.** Then the init change is not doing what the
  arithmetic predicts, and the premise needs rechecking before anything else — that number is
  pure parameterisation, no learning involved.
- **`P start` up, final metric flat.** The init is fixed but the model is the bottleneck. That
  is the point at which the architecture argument becomes the right one: multi-hypothesis /
  winner-take-all head, or distilling notebook 03's angles as supervised labels.
- **Both up but still under 0.324.** The model is still losing to a plain search, so the
  labels-from-search route is the shorter path than more model training.
- **Over 0.324.** The learned optimiser is beating the search that would have supervised it, and
  the next lever is inference budget — raise `VAL_RESTARTS` until the 600 s limit binds.

Whatever happens, `runs/submission_train.csv` in the output pane is directly submittable: the
main-stage leaderboard scores `P(ground)` on `h_train`.